In [1]:
import os
import json
import math
import random
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm.auto import tqdm

In [2]:
# config

@dataclass
class Config:
    DOMAIN: str = "books"
    DATA_PATH: str = None

    USER_COL: str = "user_id"
    ITEM_COL: str = "item_id"
    TIME_COL: str = "timestamp"

    RAW_COLS: tuple = (
        "nli_raw_plot",
        "nli_raw_characters",
        "nli_raw_writing",
    )
    
    DEDUP_USER_ITEM: bool = False
    UNMASK_TARGET_IF_SEEN: bool = True
    TEST_MASKS_VAL: bool = False

    K: int = 10

    EASE_LAMBDAS: tuple = (
        10.0, 25.0, 50.0, 100.0,
        250.0, 500.0, 1000.0, 2000.0
    )

    ALPHAS: tuple = (
        -2.0, -1.0, -0.75, -0.5, -0.35, -0.2, -0.1, -0.05, -0.01,
         0.0,
         0.01, 0.03, 0.05, 0.075, 0.1, 0.15, 0.2, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0
    )

    POOL_SIZES: tuple = (None, 20, 50, 100, 200, 500)

    SCORE_BATCH_SIZE: int = 128

    SEED: int = 42
    OUT_DIR: str = "/kaggle/working/ease_aspect_profile_reranking_diagnostic"
    SAVE_EASE_MATRIX: bool = False


CFG = Config()

In [3]:
# utils

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def find_first(patterns):
    if isinstance(patterns, str):
        patterns = [patterns]

    for pat in patterns:
        hits = list(Path("/kaggle/input").rglob(pat))
        if hits:
            return hits[0]
    return None


def find_dataset_path():
    path = find_first(["books_big_nli.parquet"])
    if path is not None:
        return str(path)

    path = find_first(["books_big.parquet"])
    if path is not None:
        return str(path)

    raise FileNotFoundError(
        "Could not find books_big_nli.parquet or books_big.parquet in /kaggle/input."
    )


def ndcg_from_rank(rank_zero_based: int) -> float:
    return 1.0 / math.log2(rank_zero_based + 2.0)


def topk_metrics_from_scores(scores_row, target, k):
    topk = np.argpartition(-scores_row, k - 1)[:k]
    topk = topk[np.argsort(-scores_row[topk])].tolist()

    if target in topk:
        rank = topk.index(target)
        hit = 1.0
        ndcg = ndcg_from_rank(rank)
    else:
        hit = 0.0
        ndcg = 0.0

    recall = hit
    precision = hit / k

    return hit, ndcg, recall, precision, topk


def aggregate_metric_rows(rows):
    return {
        "NDCG@10": float(np.mean([r["ndcg"] for r in rows])),
        "HR@10": float(np.mean([r["hit"] for r in rows])),
        "Recall@10": float(np.mean([r["recall"] for r in rows])),
        "Precision@10": float(np.mean([r["precision"] for r in rows])),
        "n_users_evaluated": int(len(rows)),
    }


def row_zscore(x):
    x = x.astype(np.float32, copy=False)
    mu = x.mean(axis=1, keepdims=True)
    sd = x.std(axis=1, keepdims=True) + 1e-8
    return (x - mu) / sd


def pool_label(pool_size):
    return "all" if pool_size is None else f"top{pool_size}"


def print_metrics(title, metrics):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    for k, v in metrics.items():
        print(f"{k:24s} {v}")


def safe_l2_normalize(x, axis=1, eps=1e-8):
    norm = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.maximum(norm, eps)


In [4]:
# load data

seed_everything(CFG.SEED)
Path(CFG.OUT_DIR).mkdir(parents=True, exist_ok=True)

if CFG.DATA_PATH is None:
    CFG.DATA_PATH = find_dataset_path()

print("DATA_PATH:", CFG.DATA_PATH)

df = pd.read_parquet(CFG.DATA_PATH)

required_cols = [CFG.USER_COL, CFG.ITEM_COL, CFG.TIME_COL] + list(CFG.RAW_COLS)
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Use books_big_nli.parquet with NLI aspect columns."
    )

df = df[required_cols].copy()
df = df.dropna(subset=[CFG.USER_COL, CFG.ITEM_COL, CFG.TIME_COL])
df[CFG.TIME_COL] = pd.to_numeric(df[CFG.TIME_COL], errors="coerce")
df = df.dropna(subset=[CFG.TIME_COL])

if CFG.DEDUP_USER_ITEM:
    df = (
        df.sort_values([CFG.USER_COL, CFG.TIME_COL, CFG.ITEM_COL])
          .drop_duplicates([CFG.USER_COL, CFG.ITEM_COL], keep="first")
          .reset_index(drop=True)
    )

print("\nLoaded data:")
print("Rows:", len(df))
print("Users:", df[CFG.USER_COL].nunique())
print("Items:", df[CFG.ITEM_COL].nunique())
print("Duplicate user-item rows:", int(df.duplicated([CFG.USER_COL, CFG.ITEM_COL]).sum()))



DATA_PATH: /kaggle/input/datasets/rita12390/ease-aspect-reranking/books_big_nli.parquet

Loaded data:
Rows: 563929
Users: 32709
Items: 2000
Duplicate user-item rows: 11020


In [5]:
# mapping

user_values = df[CFG.USER_COL].unique()
item_values = df[CFG.ITEM_COL].unique()

user2idx = {u: idx for idx, u in enumerate(user_values)}
item2idx = {i: idx for idx, i in enumerate(item_values)}
idx2user = {idx: u for u, idx in user2idx.items()}
idx2item = {idx: i for i, idx in item2idx.items()}

df["user_idx"] = df[CFG.USER_COL].map(user2idx).astype(np.int64)
df["item_idx"] = df[CFG.ITEM_COL].map(item2idx).astype(np.int64)

n_users = len(user2idx)
n_items = len(item2idx)

print("\nMapped:")
print("n_users:", n_users)
print("n_items:", n_items)


Mapped:
n_users: 32709
n_items: 2000


In [6]:
df = df.sort_values([CFG.USER_COL, CFG.TIME_COL]).copy()
df["_rank"] = df.groupby(CFG.USER_COL)[CFG.TIME_COL].rank(method="first", ascending=False)

test_df = df[df["_rank"] == 1].copy()
val_df = df[df["_rank"] == 2].copy()
train_df = df[df["_rank"] > 2].copy()

train_sequences = train_df.groupby("user_idx")["item_idx"].apply(list).to_dict()
val_targets = val_df.set_index("user_idx")["item_idx"].to_dict()
test_targets = test_df.set_index("user_idx")["item_idx"].to_dict()

eval_users = sorted(
    list(set(train_sequences.keys()) & set(val_targets.keys()) & set(test_targets.keys()))
)

train_sequences = {u: train_sequences[u] for u in eval_users}
val_targets = {u: val_targets[u] for u in eval_users}
test_targets = {u: test_targets[u] for u in eval_users}

n_eval_users = len(eval_users)
eval_user2row = {u: row for row, u in enumerate(eval_users)}
row2user = {row: u for u, row in eval_user2row.items()}

train_lengths = np.array([len(train_sequences[u]) for u in eval_users])

print("\nTemporal split:")
print("Evaluated users:", n_eval_users)
print("Mean train length:", round(float(train_lengths.mean()), 2))
print("Median train length:", round(float(np.median(train_lengths)), 2))
print("Min / Max train length:", int(train_lengths.min()), int(train_lengths.max()))

val_seen = []
test_seen = []

for u in eval_users:
    train_set = set(train_sequences[u])
    val_seen.append(val_targets[u] in train_set)
    test_seen.append(test_targets[u] in train_set)

print("\nRepeated target diagnostics:")
print("Val target already in train:", sum(val_seen), "/", len(val_seen), "=", float(np.mean(val_seen)))
print("Test target already in train:", sum(test_seen), "/", len(test_seen), "=", float(np.mean(test_seen)))




Temporal split:
Evaluated users: 32709
Mean train length: 15.24
Median train length: 11.0
Min / Max train length: 7 459

Repeated target diagnostics:
Val target already in train: 416 / 32709 = 0.012718212112874133
Test target already in train: 214 / 32709 = 0.006542541808065058


In [7]:
rows, cols = [], []

for u in eval_users:
    row = eval_user2row[u]
    for item in train_sequences[u]:
        rows.append(row)
        cols.append(item)

data = np.ones(len(rows), dtype=np.float32)

X_train = sp.csr_matrix(
    (data, (rows, cols)),
    shape=(n_eval_users, n_items),
    dtype=np.float32,
)

X_train.sum_duplicates()
X_train.data[:] = 1.0

print("\nTrain matrix:")
print("Shape:", X_train.shape)
print("Non-zero:", X_train.nnz)



Train matrix:
Shape: (32709, 2000)
Non-zero: 488228


In [8]:
item_raw_df = (
    train_df.sort_values(["item_idx", CFG.TIME_COL])
            .groupby("item_idx")[list(CFG.RAW_COLS)]
            .first()
            .reindex(range(n_items))
)

def build_item_aspect_features(item_df, cols):
    values = item_df[list(cols)].to_numpy(dtype=np.float32)
    mask = ~np.isnan(values)

    for j in range(values.shape[1]):
        valid = mask[:, j]
        if valid.sum() > 0:
            mu = values[valid, j].mean()
            sd = values[valid, j].std() + 1e-8
            values[valid, j] = (values[valid, j] - mu) / sd

        values[~valid, j] = 0.0

    return values.astype(np.float32), mask.astype(bool)


item_feat_np, item_mask_np = build_item_aspect_features(item_raw_df, CFG.RAW_COLS)

print("\nItem aspect features:")
print("Shape:", item_feat_np.shape)
print("Coverage:")
for col in CFG.RAW_COLS:
    print(f"  {col}: {item_raw_df[col].notna().mean() * 100:.1f}%")




Item aspect features:
Shape: (2000, 3)
Coverage:
  nli_raw_plot: 99.7%
  nli_raw_characters: 99.4%
  nli_raw_writing: 100.0%


In [9]:
# EASE

def fit_ease(X, reg_lambda):
    G = (X.T @ X).toarray().astype(np.float64)
    diag = np.diag_indices(G.shape[0])
    G[diag] += reg_lambda

    P = np.linalg.inv(G)
    B = -P / np.diag(P)
    B[diag] = 0.0

    return B.astype(np.float32)


def get_target_and_seen(row, split_name):
    u = row2user[row]

    if split_name == "val":
        target = val_targets[u]
        seen = set(train_sequences[u])

    elif split_name == "test":
        target = test_targets[u]
        seen = set(train_sequences[u])

        if CFG.TEST_MASKS_VAL:
            seen.add(val_targets[u])

    else:
        raise ValueError("split_name must be 'val' or 'test'.")

    if CFG.UNMASK_TARGET_IF_SEEN and target in seen:
        seen = set(x for x in seen if x != target)

    return target, seen


def evaluate_score_fn(score_fn, split_name, batch_size):
    rows_out = []

    for start in tqdm(range(0, n_eval_users, batch_size), desc=f"eval_{split_name}", leave=False):
        batch_rows = list(range(start, min(start + batch_size, n_eval_users)))
        scores = score_fn(batch_rows).astype(np.float32, copy=True)

        for local_idx, row in enumerate(batch_rows):
            target, seen = get_target_and_seen(row, split_name)

            if seen:
                scores[local_idx, list(seen)] = -1e9

            hit, ndcg, recall, precision, _ = topk_metrics_from_scores(
                scores[local_idx],
                target,
                CFG.K,
            )

            rows_out.append({
                "row": row,
                "user_idx": row2user[row],
                "target": target,
                "hit": hit,
                "ndcg": ndcg,
                "recall": recall,
                "precision": precision,
            })

    return aggregate_metric_rows(rows_out), pd.DataFrame(rows_out)


best_ease_B = None
best_ease_lambda = None
best_ease_val_ndcg = -1.0
ease_lambda_rows = []

for lam in CFG.EASE_LAMBDAS:
    print(f"\nFitting EASE lambda={lam}")
    B = fit_ease(X_train, lam)

    def ease_score_fn(batch_rows, B=B):
        return np.asarray(X_train[batch_rows] @ B, dtype=np.float32)

    val_metrics, _ = evaluate_score_fn(
        ease_score_fn,
        split_name="val",
        batch_size=CFG.SCORE_BATCH_SIZE,
    )

    ease_lambda_rows.append({
        "lambda": lam,
        **val_metrics,
    })

    print_metrics(f"EASE lambda={lam} — VAL", val_metrics)

    if val_metrics["NDCG@10"] > best_ease_val_ndcg:
        best_ease_val_ndcg = val_metrics["NDCG@10"]
        best_ease_lambda = lam
        best_ease_B = B

print("\nBest EASE lambda:", best_ease_lambda)


def ease_score_batch(batch_rows):
    return np.asarray(X_train[batch_rows] @ best_ease_B, dtype=np.float32)


ease_val, ease_val_users = evaluate_score_fn(
    ease_score_batch,
    split_name="val",
    batch_size=CFG.SCORE_BATCH_SIZE,
)

ease_test, ease_test_users = evaluate_score_fn(
    ease_score_batch,
    split_name="test",
    batch_size=CFG.SCORE_BATCH_SIZE,
)

print_metrics("EASE — VAL", ease_val)
print_metrics("EASE — TEST", ease_test)


Tuning EASE on validation...

Fitting EASE lambda=10.0


eval_val:   0%|          | 0/256 [00:00<?, ?it/s]


EASE lambda=10.0 — VAL
NDCG@10                  0.1221024116916355
HR@10                    0.19691827937264972
Recall@10                0.19691827937264972
Precision@10             0.019691827937264973
n_users_evaluated        32709

Fitting EASE lambda=25.0


eval_val:   0%|          | 0/256 [00:00<?, ?it/s]


EASE lambda=25.0 — VAL
NDCG@10                  0.12223729229080325
HR@10                    0.19722400562536305
Recall@10                0.19722400562536305
Precision@10             0.019722400562536307
n_users_evaluated        32709

Fitting EASE lambda=50.0


...

eval_val:   0%|          | 0/256 [00:00<?, ?it/s]


EASE lambda=2000.0 — VAL
NDCG@10                  0.10643961856657581
HR@10                    0.17499770705310466
Recall@10                0.17499770705310466
Precision@10             0.017499770705310464
n_users_evaluated        32709

Best EASE lambda: 25.0


eval_val:   0%|          | 0/256 [00:00<?, ?it/s]

eval_test:   0%|          | 0/256 [00:00<?, ?it/s]


EASE — VAL
NDCG@10                  0.12223729229080325
HR@10                    0.19722400562536305
Recall@10                0.19722400562536305
Precision@10             0.019722400562536307
n_users_evaluated        32709

EASE — TEST
NDCG@10                  0.0546226227092771
HR@10                    0.10660674432113486
Recall@10                0.10660674432113486
Precision@10             0.010660674432113488
n_users_evaluated        32709


In [10]:
# pure aspect-profile reranking

n_aspects = item_feat_np.shape[1]
user_aspect_profiles = np.zeros((n_eval_users, n_aspects), dtype=np.float32)

for row in range(n_eval_users):
    u = row2user[row]
    items = np.array(train_sequences[u], dtype=np.int64)

    vals = item_feat_np[items]
    masks = item_mask_np[items]

    sums = (vals * masks).sum(axis=0)
    counts = masks.sum(axis=0)

    profile = np.divide(
        sums,
        np.maximum(counts, 1),
        out=np.zeros_like(sums, dtype=np.float32),
        where=counts > 0,
    )

    user_aspect_profiles[row] = profile


user_aspect_profiles_cos = safe_l2_normalize(user_aspect_profiles)
item_feat_cos = safe_l2_normalize(item_feat_np)


def aspect_profile_dot_score_batch(batch_rows):
    U = user_aspect_profiles[batch_rows]
    return (U @ item_feat_np.T).astype(np.float32)


def aspect_profile_cos_score_batch(batch_rows):
    U = user_aspect_profiles_cos[batch_rows]
    return (U @ item_feat_cos.T).astype(np.float32)


def evaluate_ease_plus_signal_grid(
    signal_score_fn,
    signal_name,
    split_name,
    alphas,
    pool_sizes,
    batch_size=128,
):
    combos = [(float(alpha), pool) for alpha in alphas for pool in pool_sizes]
    rows_by_combo = {combo: [] for combo in combos}

    for start in tqdm(
        range(0, n_eval_users, batch_size),
        desc=f"{signal_name}_{split_name}",
        leave=False,
    ):
        batch_rows = list(range(start, min(start + batch_size, n_eval_users)))

        ease_raw = ease_score_batch(batch_rows).astype(np.float32)
        sig_raw = signal_score_fn(batch_rows).astype(np.float32)

        ease_z = row_zscore(ease_raw)
        sig_z = row_zscore(sig_raw)

        targets = []
        seen_sets = []

        for row in batch_rows:
            target, seen = get_target_and_seen(row, split_name)
            targets.append(target)
            seen_sets.append(seen)

        # EASE scores after masking, used for optional EASE top-M pools.
        ease_for_pool = ease_z.copy()

        for local_idx, seen in enumerate(seen_sets):
            if seen:
                ease_for_pool[local_idx, list(seen)] = -1e9

        pool_masks = {}

        for pool in pool_sizes:
            if pool is None:
                pool_masks[pool] = None
                continue

            M = min(int(pool), n_items)
            allowed = np.zeros_like(ease_for_pool, dtype=bool)

            topm = np.argpartition(-ease_for_pool, M - 1, axis=1)[:, :M]

            for i in range(len(batch_rows)):
                allowed[i, topm[i]] = True

            pool_masks[pool] = allowed

        for alpha, pool in combos:
            blended = ease_z + alpha * sig_z
            for local_idx, seen in enumerate(seen_sets):
                if seen:
                    blended[local_idx, list(seen)] = -1e9

            allowed = pool_masks[pool]

            if allowed is not None:
                blended = blended.copy()
                blended[~allowed] = -1e9

            for local_idx, row in enumerate(batch_rows):
                target = targets[local_idx]

                hit, ndcg, recall, precision, _ = topk_metrics_from_scores(
                    blended[local_idx],
                    target,
                    CFG.K,
                )

                rows_by_combo[(alpha, pool)].append({
                    "row": row,
                    "user_idx": row2user[row],
                    "target": target,
                    "signal": signal_name,
                    "alpha": alpha,
                    "pool": pool_label(pool),
                    "pool_size": -1 if pool is None else int(pool),
                    "hit": hit,
                    "ndcg": ndcg,
                    "recall": recall,
                    "precision": precision,
                })

    summary_rows = []

    for (alpha, pool), rows in rows_by_combo.items():
        m = aggregate_metric_rows(rows)

        summary_rows.append({
            "signal": signal_name,
            "alpha": alpha,
            "pool": pool_label(pool),
            "pool_size": -1 if pool is None else int(pool),
            **m,
        })

    summary = pd.DataFrame(summary_rows)
    summary = summary.sort_values(["NDCG@10", "HR@10"], ascending=False).reset_index(drop=True)

    return summary, rows_by_combo

In [11]:
# validation tuning

dot_val_summary, _ = evaluate_ease_plus_signal_grid(
    signal_score_fn=aspect_profile_dot_score_batch,
    signal_name="aspect_profile_dot",
    split_name="val",
    alphas=CFG.ALPHAS,
    pool_sizes=CFG.POOL_SIZES,
    batch_size=CFG.SCORE_BATCH_SIZE,
)

print(dot_val_summary.head(20).to_string(index=False))


cos_val_summary, _ = evaluate_ease_plus_signal_grid(
    signal_score_fn=aspect_profile_cos_score_batch,
    signal_name="aspect_profile_cos",
    split_name="val",
    alphas=CFG.ALPHAS,
    pool_sizes=CFG.POOL_SIZES,
    batch_size=CFG.SCORE_BATCH_SIZE,
)

print(cos_val_summary.head(20).to_string(index=False))


all_val_summary = pd.concat([dot_val_summary, cos_val_summary], ignore_index=True)
all_val_summary = all_val_summary.sort_values(["NDCG@10", "HR@10"], ascending=False).reset_index(drop=True)

best = all_val_summary.iloc[0].to_dict()

best_signal = best["signal"]
best_alpha = float(best["alpha"])
best_pool_size = None if int(best["pool_size"]) == -1 else int(best["pool_size"])

print("\n" + "=" * 80)
print("BEST PURE ASPECT RERANK CONFIG ON VALIDATION")
print("=" * 80)
print(best)

if best_signal == "aspect_profile_dot":
    best_score_fn = aspect_profile_dot_score_batch
else:
    best_score_fn = aspect_profile_cos_score_batch


Tuning EASE + pure aspect-profile DOT on validation...


aspect_profile_dot_val:   0%|          | 0/256 [00:00<?, ?it/s]


DOT validation top 20:
            signal  alpha   pool  pool_size  NDCG@10    HR@10  Recall@10  Precision@10  n_users_evaluated
aspect_profile_dot  0.050    all         -1 0.122327 0.197469   0.197469      0.019747              32709
aspect_profile_dot  0.050  top20         20 0.122327 0.197469   0.197469      0.019747              32709
aspect_profile_dot  0.050  top50         50 0.122327 0.197469   0.197469      0.019747              32709
aspect_profile_dot  0.050 top100        100 0.122327 0.197469   0.197469      0.019747              32709
aspect_profile_dot  0.050 top200        200 0.122327 0.197469   0.197469      0.019747              32709
aspect_profile_dot  0.050 top500        500 0.122327 0.197469   0.197469      0.019747              32709
aspect_profile_dot  0.075    all         -1 0.122258 0.197224   0.197224      0.019722              32709
aspect_profile_dot  0.075  top20         20 0.122258 0.197224   0.197224      0.019722              32709
aspect_profile_dot  0.

aspect_profile_cos_val:   0%|          | 0/256 [00:00<?, ?it/s]


COS validation top 20:
            signal  alpha   pool  pool_size  NDCG@10    HR@10  Recall@10  Precision@10  n_users_evaluated
aspect_profile_cos   0.05    all         -1 0.122273 0.197255   0.197255      0.019725              32709
aspect_profile_cos   0.05  top20         20 0.122273 0.197255   0.197255      0.019725              32709
aspect_profile_cos   0.05  top50         50 0.122273 0.197255   0.197255      0.019725              32709
aspect_profile_cos   0.05 top100        100 0.122273 0.197255   0.197255      0.019725              32709
aspect_profile_cos   0.05 top200        200 0.122273 0.197255   0.197255      0.019725              32709
aspect_profile_cos   0.05 top500        500 0.122273 0.197255   0.197255      0.019725              32709
aspect_profile_cos  -0.01    all         -1 0.122256 0.197438   0.197438      0.019744              32709
aspect_profile_cos  -0.01  top20         20 0.122256 0.197438   0.197438      0.019744              32709
aspect_profile_cos  -0

In [12]:
best_test_summary, best_test_rows_by_combo = evaluate_ease_plus_signal_grid(
    signal_score_fn=best_score_fn,
    signal_name=best_signal,
    split_name="test",
    alphas=(best_alpha,),
    pool_sizes=(best_pool_size,),
    batch_size=CFG.SCORE_BATCH_SIZE,
)

best_test = best_test_summary.iloc[0].to_dict()

print("\n" + "=" * 80)
print("BEST PURE ASPECT RERANK TEST")
print("=" * 80)
print(best_test_summary.to_string(index=False))


aspect_profile_dot_test:   0%|          | 0/256 [00:00<?, ?it/s]


BEST PURE ASPECT RERANK TEST
            signal  alpha pool  pool_size  NDCG@10    HR@10  Recall@10  Precision@10  n_users_evaluated
aspect_profile_dot   0.05  all         -1 0.054715 0.106974   0.106974      0.010697              32709


In [16]:
final_summary = pd.DataFrame([
    {
        "model": "EASE",
        "val_NDCG@10": ease_val["NDCG@10"],
        "val_HR@10": ease_val["HR@10"],
        "test_NDCG@10": ease_test["NDCG@10"],
        "test_HR@10": ease_test["HR@10"],
    },
    {
        "model": (
            f"EASE + aspect-profile reranking "
            f"({best_signal}, alpha={best_alpha}, pool={pool_label(best_pool_size)})"
        ),
        "val_NDCG@10": float(best["NDCG@10"]),
        "val_HR@10": float(best["HR@10"]),
        "test_NDCG@10": float(best_test["NDCG@10"]),
        "test_HR@10": float(best_test["HR@10"]),
    },
])

print(final_summary.to_string(index=False))


out_dir = Path(CFG.OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

ease_lambda_df = pd.DataFrame(ease_lambda_rows)

best_key = (best_alpha, best_pool_size)
best_test_user_metrics = pd.DataFrame(best_test_rows_by_combo[best_key])

results = {
    "config": asdict(CFG),
    "data_path": CFG.DATA_PATH,
    "n_users": int(n_users),
    "n_items": int(n_items),
    "n_eval_users": int(n_eval_users),
    "best_ease_lambda": float(best_ease_lambda),
    "ease": {
        "val": ease_val,
        "test": ease_test,
    },
    "best_rerank": {
        "signal": best_signal,
        "alpha": best_alpha,
        "pool": pool_label(best_pool_size),
        "val": best,
        "test": best_test,
    },
}

with open(out_dir / "ease_aspect_profile_reranking_results.json", "w") as f:
    json.dump(results, f, indent=2)

final_summary.to_csv(out_dir / "ease_aspect_profile_reranking_summary.csv", index=False)
ease_lambda_df.to_csv(out_dir / "ease_lambda_validation_grid.csv", index=False)

dot_val_summary.to_csv(out_dir / "ease_plus_pure_aspect_dot_val_grid.csv", index=False)
cos_val_summary.to_csv(out_dir / "ease_plus_pure_aspect_cos_val_grid.csv", index=False)
all_val_summary.to_csv(out_dir / "ease_plus_pure_aspect_all_val_grid.csv", index=False)

ease_val_users.to_csv(out_dir / "ease_val_user_metrics.csv", index=False)
ease_test_users.to_csv(out_dir / "ease_test_user_metrics.csv", index=False)
best_test_user_metrics.to_csv(out_dir / "ease_plus_pure_aspect_best_test_user_metrics.csv", index=False)

if CFG.SAVE_EASE_MATRIX:
    np.save(out_dir / "ease_B.npy", best_ease_B)

print("\nSaved files:")
for p in [
    "ease_aspect_profile_reranking_results.json",
    "ease_aspect_profile_reranking_summary.csv",
    "ease_lambda_validation_grid.csv",
    "ease_plus_pure_aspect_dot_val_grid.csv",
    "ease_plus_pure_aspect_cos_val_grid.csv",
    "ease_plus_pure_aspect_all_val_grid.csv",
    "ease_val_user_metrics.csv",
    "ease_test_user_metrics.csv",
    "ease_plus_pure_aspect_best_test_user_metrics.csv",
]:
    print(out_dir / p)

if CFG.SAVE_EASE_MATRIX:
    print(out_dir / "ease_B.npy")

                                                                     model  val_NDCG@10  val_HR@10  test_NDCG@10  test_HR@10
                                                                      EASE     0.122237   0.197224      0.054623    0.106607
EASE + aspect-profile reranking (aspect_profile_dot, alpha=0.05, pool=all)     0.122327   0.197469      0.054715    0.106974

Saved files:
/kaggle/working/ease_aspect_profile_reranking_diagnostic/ease_aspect_profile_reranking_results.json
/kaggle/working/ease_aspect_profile_reranking_diagnostic/ease_aspect_profile_reranking_summary.csv
/kaggle/working/ease_aspect_profile_reranking_diagnostic/ease_lambda_validation_grid.csv
/kaggle/working/ease_aspect_profile_reranking_diagnostic/ease_plus_pure_aspect_dot_val_grid.csv
/kaggle/working/ease_aspect_profile_reranking_diagnostic/ease_plus_pure_aspect_cos_val_grid.csv
/kaggle/working/ease_aspect_profile_reranking_diagnostic/ease_plus_pure_aspect_all_val_grid.csv
/kaggle/working/ease_aspect_profile